## CREACIÓN DE ESQUEMAS CATÁLOGOS Y VOLUMEN

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici")
catalogo = dbutils.widgets.get("catalogo")

## BAJA DE TABLAS,VOLUMEN,ESQUEMAS,CATÁLOGO

In [0]:
%sql
DROP TABLE IF EXISTS catalog_ecobici.golden.ecobici_vs_temperatura;
DROP TABLE IF EXISTS catalog_ecobici.golden.ecobici_vs_lluvia;

DROP TABLE IF EXISTS catalog_ecobici.silver.ecobici;
DROP TABLE IF EXISTS catalog_ecobici.silver.weather;

DROP TABLE IF EXISTS catalog_ecobici.bronze.ecobici;
DROP TABLE IF EXISTS catalog_ecobici.bronze.weather;

In [0]:
%sql
DROP VOLUME IF EXISTS catalog_ecobici.raw.raw_files;

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_ecobici.raw;
DROP SCHEMA IF EXISTS catalog_ecobici.bronze;
DROP SCHEMA IF EXISTS catalog_ecobici.silver;
DROP SCHEMA IF EXISTS catalog_ecobici.golden;

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_ecobici CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_ecobici;


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ${catalogo}.raw;
CREATE SCHEMA IF NOT EXISTS ${catalogo}.bronze;
CREATE SCHEMA IF NOT EXISTS ${catalogo}.silver;
CREATE SCHEMA IF NOT EXISTS ${catalogo}.golden;
CREATE VOLUME IF NOT EXISTS ${catalogo}.raw.raw_files;

## COMPROBAR VOLUMENES 

In [0]:
%sql
SHOW VOLUMES IN catalog_ecobici.raw;

## SUBIR Y VALIDAR ARCHIVOS FUENTE

In [0]:
dbutils.fs.cp(
    "/Workspace/Users/roager.mx@gmail.com/ecobiciweather-databricks-etl/datasets/ecobici2024.csv",
    "dbfs:/Volumes/catalog_ecobici/raw/raw_files/ecobici2024.csv"
)

In [0]:
display(dbutils.fs.ls("/Volumes/catalog_ecobici/raw/raw_files/"))

In [0]:
df = spark.read.csv(
    "/Volumes/catalog_ecobici/raw/raw_files/ecobici2024.csv",
    header=True,
    inferSchema=True
)

display(df)

%md
###TABLAS BRONZE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.bronze.ecobici (
  anio INT,
  mes STRING,
  fecha DATE,
  genero STRING,
  viajes INT,
  ingestion_date TIMESTAMP
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.bronze.weather (
  date DATE,
  temperature_max DOUBLE,
  temperature_avg DOUBLE,
  precipitation DOUBLE,
  ingestion_date TIMESTAMP
);

%md
###TABLAS SILVER

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.silver.ecobici (
  date DATE,
  total_viajes BIGINT,
  viajes_hombres BIGINT,
  viajes_mujeres BIGINT,
  viajes_otros BIGINT,
  ingestion_date TIMESTAMP
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.silver.weather (
  date DATE,
  temp_max DOUBLE,
  temp_avg DOUBLE,
  lluvia DOUBLE,
  ingestion_date TIMESTAMP
);

%md
###TABLAS GOLDEN

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.golden.ecobici_vs_lluvia (
  lluvia DOUBLE,
  total_viajes BIGINT,
  viajes_hombres BIGINT,
  viajes_mujeres BIGINT,
  viajes_otros BIGINT
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.golden.ecobici_vs_temperatura (
  temperatura DOUBLE,
  total_viajes BIGINT,
  viajes_hombres BIGINT,
  viajes_mujeres BIGINT,
  viajes_otros BIGINT
);